![Egeria Logo](https://raw.githubusercontent.com/odpi/egeria/main/assets/img/ODPi_Egeria_Logo_color.png)

### Coco Pharmaceuticals Labs

----

# Setting up the data hub

In this workbook, [Peter Profile](https://egeria-project.org/practices/coco-pharmaceuticals/personas/peter-profile/) is setting up the definitions for the new company data hub called `coco-data-hub`.  This data hub was introduced in the [Defining the new systems architecture](https://egeria-project.org/practices/coco-pharmaceuticals/scenarios/defining-new-systems-architecture/overview/) scenario. It groups the data stores that are used to share data between the different business units in Coco Pharmaceuticals.

<figure style="margin-left: 7%; display:inline-block;">
<img src="https://raw.githubusercontent.com/odpi/egeria-docs/main/site/docs/practices/coco-pharmaceuticals/scenarios/defining-new-systems-architecture/data-driven-systems-architecture.png">
<figcaption style="margin-left: 35%;"><strong>The role of coco-data-hub</strong></figcaption>
</figure>

<figure style="margin-left: 7%; display:inline-block; width: 20%">
<img src="https://raw.githubusercontent.com/odpi/egeria-docs/main/site/docs/practices/coco-pharmaceuticals/scenarios/defining-new-systems-architecture/peter-creating-a-data-hub.png">
<figcaption style="margin-left: 35%;"><strong>Peter At work</strong></figcaption>
</figure>


In its initial implementation, `coco-data-hub` consists of:

* A File System for storing files.
* A [PostgreSQL server](https://www.postgresql.org/) housing the operational data stores, repositories for open metadata and the tools that support the data hub along with their staging areas.
* A [Unity Catalog Server](https://unitycatalog.io/) backed by a file system (called `coco-data-lake`) and a [Delta Lake platform](https://delta.io/) called the **Data Lake**.  It is currently only used by the clinical trials team, containing the raw data collected from the hospitals along with the processed data from analysis and calculations.  The aim is to move much of the manually created analysis into an automated suit of analysis flows.  Delta Lake is a new platform for them.  If all goes well with the clinical trials pilot, they hope to bring the end-to-end analysis work for personalized medicine from order, through manufacturing to delivery, along with the accounting to this platform.  


Peter's task is to define the data hub, catalog the data stores and link them to the data hub. Once these definitions are in place, Egeria will automatically build the data dictionary for the data hub.

-----

In [ ]:

# Initialize pyegeria

import os
view_server = os.environ.get("EGERIA_VIEW_SERVER","qs-view-server")
url = os.environ.get("EGERIA_VIEW_SERVER_URL","https://host.docker.internal:9443")
user_id = "peterprofile"
user_pwd = os.environ.get("EGERIA_USER_PASSWORD")
egeria_width = 150

print("\n")
print("Accessing view server " + view_server + " on platform " + url + " for user " + user_id)
print("\n")

# These packages support the different types of markdown display
from IPython.display import display, Markdown
from pyegeria import load_mermaid, render_mermaid
load_mermaid()

# These are monitor functions from pyegeria
from commands.ops.monitor_daemon_status import display_integration_daemon_status
from commands.ops.monitor_engine_activity import display_engine_activity_c
from commands.ops.monitor_engine_status import display_gov_eng_status

import asyncio
import nest_asyncio
#nest_asyncio.apply()

from datetime import datetime
import json
import time


# The classification explorer client supports curation activities.  As such it works with most types of metadata and includes some useful query types.
from pyegeria import ClassificationExplorer
classification_explorer = ClassificationExplorer(view_server, url, user_id, user_pwd)

# The asset maker is for working with the data stores
from pyegeria import AssetMaker
asset_maker = AssetMaker(view_server, url, user_id, user_pwd)

# The collection manager client is for working with the data hub
from pyegeria import CollectionManager
collection_manager = CollectionManager(view_server, url, user_id, user_pwd)

# The automated curation client is for running the governance action processes
from pyegeria import AutomatedCuration
automated_curation = AutomatedCuration(view_server, url, user_id, user_pwd)

# The runtime manager client is for controlling Egeria's servers 
from pyegeria import RuntimeManager
runtime_manager = RuntimeManager(view_server, url, user_id, user_pwd, timeout=300)

# The governance officer client is for working with governance action processes
from pyegeria import GovernanceOfficer
governance_officer = GovernanceOfficer(view_server, url, user_id, user_pwd)


----

## Cataloguing the data managers

Peter's first task is to configure Egeria to catalog the data managers that provide the data platforms for the data hub: PostgreSQL and Unity Catalog.  Egeria has predefined [governance action processes](https://egeria-project.org/concepts/governance-action-process/) that systematically catalog all of the databases and their schemas on a regular cadence.  This means that Egeria is kept up-to-date with respect to the structure of data in these data managers.

The cataloguing is performed by the [Integration Daemon](https://egeria-project.org/concepts/integration-daemon/) server called qs-integration-daemon.  This server is running a number of integration connectors that are each specialized to perform specific actions on a particular type of technology.

The code below retrieves the unique identifier (guid) of the integration daemon to allow us to control it.

----

In [ ]:
# Retrieve the unique id (GUID) for the integration daemon.  This is used to force the cataloguing to happen immediately.
token = runtime_manager.create_egeria_bearer_token()

integration_daemon_guid=None
servers = runtime_manager.get_servers_by_name(filter_string="qs-integration-daemon")
if servers:
    for server in servers:
        if server:
            integration_daemon_guid = server["elementHeader"]["guid"]
            print(integration_daemon_guid)
        

-----

## Cataloguing the File System

From JupyterHub, the file directory used by `coco-data-hub` is called `/home/jovyan/coco-data-lake`.  Egeria sees this directory as `/deployments/coco-data-lake` and Unity Catalog sees it as `/mnt/coco-data-lake`.  This slight difference is due to the way their docker containers are built.  When we extract file names from different runtimes, we need to be sure to normalize the path of the file so that the catalog can correlate the information about the same file from different runtimes.  In this first piece of work, we are focusing on Egeria's view of the file system.  

-----

In [ ]:
# Refresh the security tokens
token = classification_explorer.create_egeria_bearer_token()
governance_officer.set_bearer_token(token)
automated_curation.set_bearer_token(token)


# Capture details of the root directory for the data hub's files

file_system_name = "egeria-workspaces"
directory_path_name = "/deployments/coco-data-lake"
directory_address = "./coco-data-lake"
directory_name = "coco-data-lake"
version_identifier = "1.0"
description = "Files in the Coco Pharmaceuticals Data Lake."


# Display the process
create_and_catalog_process_name="FileDirectory::CreateAsCatalogTargetGovernanceActionProcess"

process_guid = classification_explorer.get_element_guid_by_unique_name(create_and_catalog_process_name)
process_graph = governance_officer.get_governance_process_graph(process_guid, output_format="JSON")
render_mermaid(process_graph.get("governanceActionProcessMermaidGraph"))

# Register the directory for cataloguing (run the process)
print("Starting the " + create_and_catalog_process_name + " process")

request_parameters = {
    "fileSystemName" : file_system_name,
    "directoryPathName" : directory_path_name,
    "directoryAddress" : directory_address,
    "directoryName" : directory_name,
    "versionIdentifier" : version_identifier,
    "description" : description,
    "waitForDirectory" : True
}

instance_guid = automated_curation.initiate_gov_action_process(action_type_qualified_name=create_and_catalog_process_name, 
                                                               request_source_guids=None, 
                                                               action_targets=None, 
                                                               start_time=datetime.now(),
                                                               request_parameters=request_parameters, 
                                                               orig_service_name=None, 
                                                               orig_engine_name=None)



----

The command below shows the status of the governance action.  Run (and re-run) the cell below until you see the status "COMPLETED".  You may see statuses such as "PROPOSED", "APPROVED", "IN_PROGRESS" but not "FAILED".

----

In [ ]:

token = governance_officer.create_egeria_bearer_token()

# Show the execution path of the process
process_graph = governance_officer.get_governance_process_graph(instance_guid)
render_mermaid(process_graph.get("governanceActionProcessMermaidGraph"))


-----

Now the integration connector is refreshed.  This results in a complete scan of the contents of the directory which are then catalogued in Egeria.

----

In [ ]:
token = runtime_manager.create_egeria_bearer_token()

# Refresh the files integration connector
# An asynchronous call is used since this will return once the refresh is complete - which can result in a timeout on a normal synchronous request.

files_cataloguer="FilesCataloguer"
print("Refreshing: " + files_cataloguer)
await runtime_manager._async_refresh_integration_connector(server_guid=integration_daemon_guid,connector_name=files_cataloguer)

print("Completed.")


----

When we check the status of the connector you can see the directory is listed as being monitored.

----

In [ ]:
display_integration_daemon_status([files_cataloguer], 
                                  integ_server="Coco Pharmaceuticals.qs-integration-daemon", 
                                  paging=True, 
                                  width=120)

----

### Catalog the PostgreSQL Server

In the code below, the governance action process that sets up cataloguing of the PostgreSQL Server is displayed and then run.  Next the execution path of the process is displayed, Finally, the integration daemon is called to prompt the connecters performing the cataloguing to run now rather than waiting for the next refresh period.  This final stage is not strictly necessary for normal operation - but it makes the results available quickly which is convenient for this workbook. 

----

In [ ]:
# Refresh the security tokens
token = classification_explorer.create_egeria_bearer_token()
governance_officer.set_bearer_token(token)
automated_curation.set_bearer_token(token)


# Capture the details of the PostgreSQL Server
postgres_host_identifier="host.docker.internal"
postgres_port_number="5442"
postgres_server_name="Coco PostgreSQL Server 1"
postgres_secrets_collection_name = "PostgreSQL Server Secret"
secrets_store_path_name = "secrets/integration.omsecrets"

# Display the process
create_and_catalog_server_name="PostgreSQLServer::CreateAsCatalogTargetGovernanceActionProcess"

process_guid = classification_explorer.get_element_guid_by_unique_name(create_and_catalog_server_name)
print(process_guid)
process_graph = governance_officer.get_governance_process_graph(process_guid, output_format="JSON")
render_mermaid(process_graph.get("governanceActionProcessMermaidGraph"))

# Register the PostgreSQL Server for cataloguing (run the process)
print("Starting the " + create_and_catalog_server_name + " process")

request_parameters = {
    "serverName" : postgres_server_name,
    "hostIdentifier" : postgres_host_identifier,
    "portNumber" : postgres_port_number,
    "secretsStorePathName" : secrets_store_path_name,
    "secretsCollectionName" : postgres_secrets_collection_name,
    "versionIdentifier" : "1.0",
    "description" : "PostgreSQL Server running in the shared infrastructure of egeria-workspaces."
}
instance_guid = automated_curation.initiate_gov_action_process(action_type_qualified_name=create_and_catalog_server_name, 
                                                               request_source_guids=None, 
                                                               action_targets=None, 
                                                               start_time=datetime.now(),
                                                               request_parameters=request_parameters, 
                                                               orig_service_name=None, 
                                                               orig_engine_name=None)


----

The command below shows the status of the governance action.  Run (and re-run) the cell below until you see the status "COMPLETED".  You may see statuses such as "PROPOSED", "APPROVED", "IN_PROGRESS" but not "FAILED".

----

In [ ]:

token = governance_officer.create_egeria_bearer_token()

# Show the execution path of the process
process_graph = governance_officer.get_governance_process_graph(instance_guid)
render_mermaid(process_graph.get("governanceActionProcessMermaidGraph"))


-----

Now the integration connectors are refreshed.  This results in a complete scan of the contents of the PostgreSQL Server which are then catalogued in Egeria.

----

In [ ]:
token = runtime_manager.create_egeria_bearer_token()

# Refresh the server integration connector
# An asynchronous call is used since this will return once the refresh is complete - which can result in a timeout on a normal synchronous request.

postgres_server_cataloguer="PostgreSQLServerCataloguer"
print("Refreshing: " + postgres_server_cataloguer)
await runtime_manager._async_refresh_integration_connector(server_guid=integration_daemon_guid,connector_name=postgres_server_cataloguer)


# Refresh the database integration connector
# An asynchronous call is used since this will return once the refresh is complete - which can result in a timeout on a normal synchronous request.

postgres_database_cataloguer="JDBCDatabaseCataloguer"
print("Refreshing: " + postgres_database_cataloguer)
await runtime_manager._async_refresh_integration_connector(server_guid=integration_daemon_guid,connector_name=postgres_database_cataloguer)

print("Completed.")

-----

When we check the status of the connectors, you can see that the connectors are configured and busy processing the elements of the server.  

The connectors are split this way because the SQL standard allows a Java-based tool such as Egeria to extract the schemas, tables and columns for any type of relational database through the JDBC interface.  However when it comes to retrieving the list of databases on a database server, this is not covered by the standard and so each database vendor does it differently and we need a vender-specific implementation.

Perhaps more importantly, the resulting assets catalogued in open metadata are for different audiences: the PostgreSQL server is of interest to the infrastructure team and and databases and schemas are of interest to the data engineers and data scientists.  Each connector supports the different interest.

-----

In [ ]:
display_integration_daemon_status([postgres_server_cataloguer, postgres_database_cataloguer], 
                                  integ_server="Coco Pharmaceuticals.qs-integration-daemon", 
                                  paging=True, 
                                  width=120)

----

## Cataloguing Unity Catalog

Now we perform the same steps for Unity Catalog.  Unity Catalog provides the same level of cataloguing that a PostgreSQL Server provides.  We call them **Data Manager Catalogs**. The difference is the PosgreSQL manages the storage of thedata, where as Unity catalog describes heterogeneous data stores.

At the top level, Unity Catalog has catalogs - like PostgreSQL databases - schemas, and then tables with columns.  It also defines functions - like PostgreSQL stored procedures and user defined functions (UDFs).  Finally, unlike a database, it has data volumes - these are directories of files - often raw data input or unstructured data.  

The aim of Unity Catalog is to draw together data using different storage platforms and formats into common data processing concepts. This is the value of using Unity Catalog in the data hub since it manages the cataloguing for a diverse range of data.  Egeria has a symbiotic relationship with Unity Catalog: it can extract catalogued data descriptions for open metadata, as well as provision data descriptions from open metadata into Unity Catalog.

----

In [ ]:
# Refresh the security tokens
token = classification_explorer.create_egeria_bearer_token()
governance_officer.set_bearer_token(token)
automated_curation.set_bearer_token(token)


# Capture the details of the Unity Catalog Server
uc_technology_type = "Unity Catalog Server"
uc_host_url="http://host.docker.internal"
uc_port_number="8087"
uc_server_name="Coco Unity Catalog"
uc_server_description="Coco's instance of the Unity Catalog (UC) Server."
uc_version="v0.4.0"
uc_server_user_id="uc1"
uc_server_qualified_name=uc_technology_type + "::" + uc_server_name
uc_server_network_address=uc_host_url + ":" + uc_port_number

# Display the process
create_and_catalog_server_name="UnityCatalogServer::CreateAsCatalogTargetGovernanceActionProcess"

process_guid = classification_explorer.get_element_guid_by_unique_name(create_and_catalog_server_name)
print(process_guid)
process_graph = governance_officer.get_governance_process_graph(process_guid, output_format="JSON")
render_mermaid(process_graph.get("governanceActionProcessMermaidGraph"))

# Register the Unity Catalog Server for cataloguing (run the process)
print("Starting the " + create_and_catalog_server_name + " process")

request_parameters = {
    "hostURL" : uc_host_url,
    "portNumber" : uc_port_number,
    "serverName" : uc_server_name,  
    "resourceName" : uc_server_network_address,  
    "versionIdentifier" : uc_version,
    "description" : uc_server_description,
    "serverUserId" : uc_server_user_id
}

instance_guid = automated_curation.initiate_gov_action_process(action_type_qualified_name=create_and_catalog_server_name, 
                                                        request_source_guids=None, 
                                                        action_targets=None, 
                                                        start_time=datetime.now(),
                                                        request_parameters=request_parameters, 
                                                        orig_service_name=None, 
                                                        orig_engine_name=None)


----

The command below shows the status of the governance action.  Run (and re-run) the cell below until you see the status "COMPLETED".  You may see statuses such as "PROPOSED", "APPROVED", "IN_PROGRESS" but not "FAILED".

----

In [ ]:
token = governance_officer.create_egeria_bearer_token()

# Show the execution path of the process
process_graph = governance_officer.get_governance_process_graph(instance_guid)
render_mermaid(process_graph.get("governanceActionProcessMermaidGraph"))

-----

Now the integration connectors are refreshed.  This results in a complete scan of the contents of Unity Catalog which are then catalogued in Egeria.

----

In [ ]:
token = runtime_manager.create_egeria_bearer_token()

# Refresh the server integration connector
# An asynchronous call is used since this will return once the refresh is complete - which can result in a timeout on a normal synchronous request.
uc_server_cataloguer="UnityCatalogServerSynchronizer"
print("Refreshing: " + uc_server_cataloguer)
await runtime_manager._async_refresh_integration_connector(server_guid=integration_daemon_guid,connector_name=uc_server_cataloguer)

# Refresh the inside catalog integration connector
# An asynchronous call is used since this will return once the refresh is complete - which can result in a timeout on a normal synchronous request.
uc_catalog_cataloguer="UnityCatalogInsideCatalogSynchronizer"
print("Refreshing: " + uc_catalog_cataloguer)
await runtime_manager._async_refresh_integration_connector(server_guid=integration_daemon_guid,connector_name=uc_catalog_cataloguer)

print("Completed.")

----

When we view the Unity Catalog connectors, we see the same division of work between them as with PostgreSQL.

----

In [ ]:
display_integration_daemon_status([uc_server_cataloguer, uc_catalog_cataloguer], 
                                  integ_server="Coco Pharmaceuticals.qs-integration-daemon", 
                                  paging=True, 
                                  width=130)

----

## Creating the Data Hub

The Data Hub element is a special type of collection.  The data stores will be added as members in subsequent steps.

----

In [ ]:
token = collection_manager.create_egeria_bearer_token()
classification_explorer.set_bearer_token(token)
governance_officer.set_bearer_token(token)

# Check to see if the data hub exists already
data_hub_guid=None
data_hub_name="coco-data-hub"

data_hubs=collection_manager.get_collections_by_name(name = data_hub_name, metadata_element_type_name = "DataHub", graph_query_depth=0)

if data_hubs:
    if type(data_hubs) == str:
        print(data_hubs)
    else:
        for data_hub in data_hubs:
            if data_hub:
                data_hub_guid = data_hub['elementHeader']['guid']
                print("Retrieved existing data hub: " + data_hub_guid)

if data_hub_guid == None:
    data_hub_guid=collection_manager.create_collection(display_name=data_hub_name,
                                                       description="This data hub groups the data stores that are used to share data between the different business units in Coco Pharmaceuticals.",
                                                       category="Shared Infrastructure",
                                                       initial_classifications=None,
                                                       prop=["DataHub"])
    print("Created new data hub: " + data_hub_guid);

    # Link the new data hub to the list of strategic data hubs called "RootCollection::Coco::Strategic Data Hubs" - 90725652-7878-4f32-aef9-be6a94670411
    strategic_data_hubs_guid = classification_explorer.get_element_guid_by_unique_name("RootCollection::Coco::Strategic Data Hubs")

    if strategic_data_hubs_guid != "No elements found":
        print("Linking new data hub to the strategic data hubs")
        collection_manager.add_to_collection(collection_guid=strategic_data_hubs_guid, element_guid=data_hub_guid)

    # Link the new data hub to the data hub solution component called "CocoPharma::SolutionComponent::DataHub" - this is loaded through the solution-design.md
    data_hub_solution_component_guid = classification_explorer.get_element_guid_by_unique_name("CocoPharma::SolutionComponent::DataHub")

    if data_hub_solution_component_guid != "No elements found":
        body={
            "class" : "NewRelationshipRequestBody",
            "properties" : {
               "class" : "ImplementedByProperties",
               "designStep" : "no-regrets",
               "role" : "metadata-description",
               "description" : "The data hub provides a root element in open metadata that is used by the data hub manager to control the management of data stores attached to the data hub."
            }
        }
        
        print("Linking new data hub as implementation to the data hub component")
        governance_officer.link_design_to_implementation(design_desc_guid=data_hub_solution_component_guid, implementation_guid=data_hub_guid, body=body)


    
    

----

Data Hubs are managed by the *Data Hub Manager* connector.  This connector monitors the data stores attached to the data hub and build a data dictionary of the types of data it finds.

----

In [ ]:
token = runtime_manager.create_egeria_bearer_token()

# Refresh the database integration connector
# An asynchronous call is used since this will return once the refresh is complete - which can result in a timeout on a normal synchronous request.

data_hub_manager="DataHubManager"
print("Refreshing: " + data_hub_manager)
await runtime_manager._async_refresh_integration_connector(server_guid=integration_daemon_guid,connector_name=data_hub_manager)

print("Completed.")

----

Once the refresh is complete, you should see that the data hub manager is now processing `coco-data-hub`.

----

In [ ]:

display_integration_daemon_status(["DataHubManager"], 
                                  integ_server="Coco Pharmaceuticals.qs-integration-daemon", 
                                  paging=True, 
                                  width=130)



-----

Data hub manager begins building a data dictionary for `coco-data-hub`.  At this point, there is not much to work on.  We need to add data stores to the data hub for it to become more interesting.

----

In [ ]:
token = collection_manager.create_egeria_bearer_token()

data_hub_report = collection_manager.get_collection_by_guid(guid=data_hub_guid,
                                                            output_format="MERMAID",
                                                            include_only_relationships=["CatalogTarget", "DataDescription"],
                                                            report_spec="Collections")
display(Markdown(data_hub_report))


-----

Next Peter needs to locate the data stores for the data hub, beginning with the `coco-data-lake`.

## Locating the coco-data-lake File Directory

The unique identifier for the `coco-data-lake` asset is located and used to link it to the data hub.

-----

In [ ]:

token = collection_manager.create_egeria_bearer_token()
classification_explorer.set_bearer_token(token)

coco_data_lake_qualified_name = "File System Directory::egeria-workspaces:/deployments/coco-data-lake"

coco_data_lake_guid=classification_explorer.get_element_guid_by_unique_name(coco_data_lake_qualified_name)
print(coco_data_lake_guid)
if coco_data_lake_guid != "No elements found":
    collection_manager.add_to_collection(collection_guid=data_hub_guid,
                                         element_guid=coco_data_lake_guid)
    

-----

Now the Unity Catalog content is linked to the data hub.

## Locating the Unity Catalog clinical-trials catalog

Peter first creates an asset that describes the PostgreSQL database called `coco_pharma`.  This is also known as *cataloguing the PostgreSQL database*.  This is linked to two more assets for each of the database schemas: `coco_ods` and `coco_sus`.  

This process uses [templates](https://egeria-project.org/concepts/template/) to ensure the connections to the database are correctly set up.

----

In [ ]:
token = collection_manager.create_egeria_bearer_token()
classification_explorer.set_bearer_token(token)

uc_clinical_trials_catalog_qualified_name="Unity Catalog Catalog::" + uc_host_url + ":" + uc_port_number + "::clinical_trials"

uc_clinical_trials_catalog_guid=classification_explorer.get_element_guid_by_unique_name(uc_clinical_trials_catalog_qualified_name)
print(uc_clinical_trials_catalog_guid)
if uc_clinical_trials_catalog_guid != "No elements found":
    collection_manager.add_to_collection(collection_guid=data_hub_guid,
                                         element_guid=uc_clinical_trials_catalog_guid)
    

-----

Finally Peter needs to locate the data stores located on the PostgreSQL Server.

## Locating the PostgreSQL Schemas

Peter now needs to link in the `coco-ods` and `coco-sus` database schemas from the `coco-pharma` database.

----

In [ ]:
token = collection_manager.create_egeria_bearer_token()
classification_explorer.set_bearer_token(token)

postgres_coco_ods_schema_qualified_name="PostgreSQL Relational Database::" + postgres_server_name + "::coco_pharma::coco_ods"
print(postgres_coco_ods_schema_qualified_name)
postgres_coco_sus_schema_qualified_name="PostgreSQL Relational Database::" + postgres_server_name + "::coco_pharma::coco_sus"
print(postgres_coco_sus_schema_qualified_name)


postgres_coco_ods_schema_guid=classification_explorer.get_element_guid_by_unique_name(postgres_coco_ods_schema_qualified_name)
print(postgres_coco_ods_schema_guid)
if postgres_coco_ods_schema_guid != "No elements found":
    collection_manager.add_to_collection(collection_guid=data_hub_guid,
                                         element_guid=postgres_coco_ods_schema_guid)

postgres_coco_sus_schema_guid=classification_explorer.get_element_guid_by_unique_name(postgres_coco_sus_schema_qualified_name)
print(postgres_coco_sus_schema_guid)
if postgres_coco_sus_schema_guid != "No elements found":
    collection_manager.add_to_collection(collection_guid=data_hub_guid,
                                         element_guid=postgres_coco_sus_schema_guid)


----

## In summary

This notebook created a new data hub and registered postgres databases and unity catalog with the data hub.


In [ ]:
token = collection_manager.create_egeria_bearer_token()

data_hub_report = collection_manager.get_collection_by_guid(guid=data_hub_guid,
                                                            output_format="REPORT",
                                                            report_spec="Collections")
display(Markdown(data_hub_report))


-----

Below is the solution blueprint for the data hub.  At this point Peter has created the component in the center.  The next step in the project is to build the data pipelines that pump data in and out of the data hub.

-----

In [ ]:
token = classification_explorer.create_egeria_bearer_token()

blueprint_qualified_name="CocoPharma::SolutionBlueprint::DataDrivenSystemsArchitecture"

solution_blueprint=egeria_client.get_element_by_unique_name(blueprint_qualified_name)
render_mermaid(solution_blueprint.get("solutionBlueprintMermaidGraph"))
